# Disagreement: good learning with nonzero loss

Day 15 · Chapter 10. CPU, authored synthetic data, no downloads. Predict before each reference, change one declared variable, and interpret the result. Code is complete so the notebook runs from beginning to end. Completing its execution is material verification, not an assessment of learner mastery.

[Canonical chapter](../../book/chapters/10-preferences-and-reward-models.md)


In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
while not (root / "src" / "dongxi_llms").is_dir():
    if root.parent == root:
        raise RuntimeError("Open this notebook within the Dongxi_LLMs repository")
    root = root.parent
sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white",
                     "font.family": "DejaVu Sans", "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
blue, orange, green = "#356a9b", "#b97139", "#3d8268"



## Exercise 1: conflicting votes

Seven of ten repeated judgments prefer the first answer. Predict whether the best score margin should be infinite, zero, or finite. Explain why zero expected gradient need not mean zero loss.


In [ ]:
from dongxi_llms.reward_model_lab import bt_loss,calibration
q = torch.tensor(.7,dtype=torch.float64)
margin = torch.zeros((),dtype=torch.float64,requires_grad=True)
history = []
for step in range(120):
    loss = bt_loss(margin,torch.zeros_like(margin),q)
    gradient, = torch.autograd.grad(loss,margin)
    with torch.no_grad(): margin -= .5*gradient
    history.append((float(loss.detach()),float(margin.detach().sigmoid())))
optimum = torch.log(q/(1-q))
entropy = -q*q.log()-(1-q)*(1-q).log()
print({"learned_margin":float(margin.detach()),"analytic_margin":float(optimum),
       "minimum_NLL":float(entropy),"final_probability":float(margin.detach().sigmoid())})
fig, axes = plt.subplots(1,2,figsize=(10,3.5))
axes[0].plot([x[0] for x in history],color=blue);axes[0].axhline(float(entropy),color=orange,ls="--")
axes[0].set(xlabel="gradient update",ylabel="expected preference NLL (nats)")
axes[1].plot([x[1] for x in history],color=blue);axes[1].axhline(float(q),color=orange,ls="--")
axes[1].set(xlabel="gradient update",ylabel="P(first wins)",ylim=(0,1))
plt.tight_layout();plt.show()


![Saved reference plot for 02 disagreement and calibration, figure 1](../figures/chapter-10/day-15-02_disagreement_and_calibration-01.png)

Saved reference output from the verified CPU run. Run the preceding cell to regenerate the current experiment; this image does not change when parameters are edited.


**Solution:** the finite margin is log(7/3). Opposing labels balance in expectation at probability .7, leaving vote entropy. The dashed lines are analytical targets, not fitted trend lines.

## Exercise 2: keep ranks, change confidence

Stretch every true margin by 3. Predict the effect on ranking and probability quality. This simulator knows the population probabilities, so it can show calibration without sampling labels.


In [ ]:
true_margin = torch.linspace(-3,3,400,dtype=torch.float64)
truth = true_margin.sigmoid()
fig, ax = plt.subplots(figsize=(5.5,4))
ax.plot([0,1],[0,1],ls="--",color=".65",label="calibrated")
results = {}
for scale in (1.,3.):
    p = (scale*true_margin).sigmoid()
    result = calibration(p,truth,bins=8); results[scale]=result
    ax.plot([b[1] for b in result["bins"]],[b[2] for b in result["bins"]],
            marker="o",label=f"margin scale {scale:g}")
    assert torch.equal(p>.5,truth>.5)
ax.set(xlabel="mean predicted preference probability",ylabel="mean true preference probability",
       xlim=(0,1),ylim=(0,1));ax.legend();plt.show()
print({scale:{k:v for k,v in r.items() if k!="bins"} for scale,r in results.items()})


![Saved reference plot for 02 disagreement and calibration, figure 2](../figures/chapter-10/day-15-02_disagreement_and_calibration-02.png)

Saved reference output from the verified CPU run. Run the preceding cell to regenerate the current experiment; this image does not change when parameters are edited.


**Read the figure:** the stretched model is overconfident even though every rank is unchanged. Expected binary Brier includes Bernoulli variance; it is not simply squared distance to the soft target.

## Exercise 3: repair confidence, then identify a limit

Predict which positive temperature reverses the margin stretch. Would that also repair wrong rankings?


In [ ]:
stretched = 3*true_margin
temperature = 3.
repaired = (stretched/temperature).sigmoid()
torch.testing.assert_close(repaired,truth)
wrong_ranking = (-true_margin/temperature).sigmoid()
print({"repair_max_error":float((repaired-truth).abs().max()),
       "wrong_ranking_expected_brier":calibration(wrong_ranking,truth)["expected_brier"]})


**Solution:** temperature 3 undoes this known construction, but positive temperature cannot reverse rankings. In real data, fit temperature on a calibration split and score it on a separate final split. Here we used known simulator parameters; that is an analytical demonstration, not evidence of held-out human calibration.

**Discussion:** disagreement can express legitimate different rubrics. A half-win target cannot distinguish ties from uncertainty or abstention.
